# Simple Feature Engineering: Project Walkthrough

This notebook is a top-to-bottom tour of the project. The heavy lifting lives in
`src/feature_engineering/`; here we just call the public API on small toy data so
the moving parts are easy to see.

We cover:

1. The one-minute mental model
2. The feature menu (the registry)
3. Toy OHLCV data and cleaning
4. Batch features with `compute_features`
5. The cached `FeatureEngine` wrapper (research / backtest)
6. The incremental `OnlineFeatureEngine` (live, O(1) per bar)

## 1) The project in one minute

The pipeline is one small, linear workflow:

```text
validate config -> load data -> clean invalid rows -> compute features -> export files
```

- Features are grouped by **category**: `returns`, `trend`, `volatility`, `volume`, `target`.
- Each feature is computed **per symbol** (and optionally per session) so one
  ticker's history never leaks into another's.
- The `target` category is forward-looking and is normally excluded from live
  feature sets.

In [1]:
import numpy as np
import pandas as pd

from feature_engineering import (
    REGISTRY,
    clean_ohlcv,
    compute_features,
    FeatureEngine,
    OnlineFeatureEngine,
)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
print(f"{len(REGISTRY)} features registered")

18 features registered


## 2) The feature menu (registry)

Every feature registers itself with a category, a lookback, and a plain-language
formula. This table is the menu you choose from in `config.toml`.

In [2]:
# Example params so lookbacks that depend on a window/span resolve to a number.
example_params = {"window": 14, "periods": 20, "fast": 12, "slow": 26, "signal": 9, "bars": 1}

menu = pd.DataFrame(
    [
        {
            "fn": name,
            "category": spec.category,
            "lookback": spec.resolve_lookback(example_params),
            "calculation": spec.calculation,
        }
        for name, spec in sorted(REGISTRY.items(), key=lambda kv: (kv[1].category, kv[0]))
    ]
)
menu

,fn,category,lookback,calculation
0,log_return,returns,1,ln(close_t / close_{t-1})
1,simple_return,returns,1,close_t / close_{t-1} - 1
2,next_n_bar_return,target,0,close_{t+bars} / close_t - 1
3,macd_histogram,trend,35,"MACD_line - EMA(MACD_line, signal)"
4,macd_line,trend,26,"EMA(close, fast) - EMA(close, slow)"
5,macd_signal,trend,35,"EMA(MACD_line, signal)"
6,moving_average,trend,14,mean(close) over trailing window
7,price_vs_sma,trend,14,close_t / moving_average_t - 1
8,rate_of_change,trend,20,close_t / close_{t-periods} - 1
9,relative_strength_index,trend,14,"100 - 100 / (1 + avg_gain / avg_loss), Wilder-..."


## 3) Toy OHLCV data

Two symbols of one-minute bars across two trading days. Intraday bars are exactly
the case where windows must not cross the overnight gap, which we use later.

In [3]:
def make_symbol(symbol: str, start_price: float, day: str, seed: int, n: int = 14) -> pd.DataFrame:
    """Deterministic single-symbol, single-day minute bars."""
    rng = np.random.default_rng(seed)
    close = start_price * np.exp(np.cumsum(rng.normal(0.0, 0.004, n)))
    high = close * (1.0 + rng.uniform(0.0, 0.004, n))
    low = close * (1.0 - rng.uniform(0.0, 0.004, n))
    open_ = np.clip(close * (1.0 + rng.normal(0.0, 0.002, n)), low, high)
    volume = rng.integers(1_000, 4_000, n).astype(float)
    ts = pd.date_range(f"{day} 09:30:00", periods=n, freq="min")
    return pd.DataFrame(
        {"symbol": symbol, "ts": ts, "open": open_, "high": high, "low": low,
         "close": close, "volume": volume}
    )


raw = pd.concat(
    [
        make_symbol("AAPL", 100.0, "2024-01-02", seed=1),
        make_symbol("AAPL", 102.0, "2024-01-03", seed=2),
        make_symbol("MSFT", 300.0, "2024-01-02", seed=3),
        make_symbol("MSFT", 305.0, "2024-01-03", seed=4),
    ],
    ignore_index=True,
)
raw.head()

,symbol,ts,open,high,low,close,volume
0,AAPL,2024-01-02 09:30:00,100.073980,100.259775,100.073980,100.138329,1659.0
1,AAPL,2024-01-02 09:31:00,100.453496,100.650221,100.078187,100.467973,3565.0
2,AAPL,2024-01-02 09:32:00,100.410768,100.654793,100.393186,100.600854,3002.0
3,AAPL,2024-01-02 09:33:00,100.058154,100.239194,100.031441,100.077823,3583.0
4,AAPL,2024-01-02 09:34:00,100.460085,100.522645,100.190409,100.440904,3520.0


## 4) Cleaning

`clean_ohlcv` drops only rows that violate basic market-data invariants and
returns a report. Our toy data is already clean, so nothing is dropped.

In [4]:
cleaned, report = clean_ohlcv(raw)
print("rows:", report["initial_rows"], "->", report["final_rows"])
report["rules"]

rows: 56 -> 56


{'drop_missing_numeric_values': {'enabled': True,
  'dropped': 0,
  'reason': 'open, high, low, close, and volume must be present'},
 'drop_zero_prices': {'enabled': True,
  'dropped': 0,
  'reason': 'open, high, low, and close must be positive prices'},
 'drop_high_lt_low': {'enabled': True,
  'dropped': 0,
  'reason': 'high must be greater than or equal to low'},
 'drop_ohlc_violations': {'enabled': True,
  'dropped': 0,
  'reason': 'open and close must sit inside the low-high range'}}

## 5) Batch features with `compute_features`

We pick a few features per category and turn on `reset_by_session` because this
is intraday data. The forward `next_n_bar_return` **target** is included here for
training data; in live use you would exclude it.

In [5]:
demo_config = {
    "features": {
        "reset_by_session": True,  # intraday: do not cross the overnight gap
        "params": [
            {"name": "log_return", "fn": "log_return"},
            {"name": "fwd_1bar", "fn": "next_n_bar_return", "bars": 1},   # target
            {"name": "ma_3", "fn": "moving_average", "window": 3},
            {"name": "rsi_5", "fn": "relative_strength_index", "window": 5},
            {"name": "atr_5", "fn": "average_true_range", "window": 5},
            {"name": "macd_hist", "fn": "macd_histogram", "fast": 3, "slow": 6, "signal": 2},
            {"name": "vwap", "fn": "vwap"},
            {"name": "price_vs_vwap", "fn": "price_vs_vwap"},
        ],
    }
}

features = compute_features(cleaned, demo_config)
features.head(8)

,symbol,ts,log_return,fwd_1bar,ma_3,rsi_5,atr_5,macd_hist,vwap,price_vs_vwap
0,AAPL,2024-01-02 09:30:00,NaN,0.003292,NaN,NaN,NaN,NaN,100.157361,-0.000190
1,AAPL,2024-01-02 09:31:00,0.003286,0.001323,NaN,NaN,NaN,NaN,100.322121,0.001454
2,AAPL,2024-01-02 09:32:00,0.001322,-0.005199,100.402385,NaN,NaN,NaN,100.405141,0.001949
3,AAPL,2024-01-02 09:33:00,-0.005213,0.003628,100.382216,NaN,NaN,NaN,100.317458,-0.002389
4,AAPL,2024-01-02 09:34:00,0.003621,0.001787,100.373194,NaN,0.348234,NaN,100.332888,0.001077
5,AAPL,2024-01-02 09:35:00,0.001785,-0.002146,100.379710,78.373851,0.362223,NaN,100.374715,0.002448
6,AAPL,2024-01-02 09:36:00,-0.002148,0.002327,100.488609,66.738607,0.399289,-0.013961,100.379170,0.000253
7,AAPL,2024-01-02 09:37:00,0.002324,0.001459,100.554367,72.301886,0.415859,0.004998,100.397155,0.002401


In [6]:
# Feature health: leading NaNs are the warmup period of each rolling window.
features.drop(columns=["symbol", "ts"]).isna().mean().round(3).rename("null_fraction").to_frame()

,null_fraction
log_return,0.071
fwd_1bar,0.071
ma_3,0.143
rsi_5,0.357
atr_5,0.286
macd_hist,0.429
vwap,0.000
price_vs_vwap,0.000


## 6) The cached `FeatureEngine` (research / backtest)

`FeatureEngine` validates and resolves the feature list **once** in its
constructor, then reuses it on every `transform` call. Same numbers as
`compute_features`, but no per-call config walking. This is the convenient entry
point for a backtest loop that calls `transform` many times.

In [7]:
engine = FeatureEngine(demo_config)
print("engine output columns:", engine.feature_names)

batch_again = engine.transform(cleaned)
# Identical to the one-shot function.
pd.testing.assert_frame_equal(batch_again, features)
print("FeatureEngine.transform == compute_features: OK")

engine output columns: ['log_return', 'fwd_1bar', 'ma_3', 'rsi_5', 'atr_5', 'macd_hist', 'vwap', 'price_vs_vwap']
FeatureEngine.transform == compute_features: OK


## 7) The incremental `OnlineFeatureEngine` (live, O(1) per bar)

For a live bot, recomputing the whole history every bar is wasteful. The online
engine keeps a little state per symbol and updates each feature in O(1) per bar
via `update(bar)`. It refuses forward-looking targets, so we drop `fwd_1bar`.

In [8]:
live_config = {
    "features": {
        "reset_by_session": True,
        "params": [p for p in demo_config["features"]["params"] if p["fn"] != "next_n_bar_return"],
    }
}

online = OnlineFeatureEngine(live_config)

# Feed the first few AAPL bars one at a time, exactly as a live feed would.
aapl_bars = cleaned[cleaned["symbol"] == "AAPL"].sort_values("ts").to_dict("records")
for bar in aapl_bars[:6]:
    values = online.update(bar)
    pretty = {k: (round(v, 4) if v == v else None) for k, v in values.items()}
    print(bar["ts"].strftime("%m-%d %H:%M"), pretty)

01-02 09:30 {'log_return': None, 'ma_3': None, 'rsi_5': None, 'atr_5': None, 'macd_hist': None, 'vwap': 100.1574, 'price_vs_vwap': -0.0002}
01-02 09:31 {'log_return': 0.0033, 'ma_3': None, 'rsi_5': None, 'atr_5': None, 'macd_hist': None, 'vwap': 100.3221, 'price_vs_vwap': 0.0015}
01-02 09:32 {'log_return': 0.0013, 'ma_3': 100.4024, 'rsi_5': None, 'atr_5': None, 'macd_hist': None, 'vwap': 100.4051, 'price_vs_vwap': 0.0019}
01-02 09:33 {'log_return': -0.0052, 'ma_3': 100.3822, 'rsi_5': None, 'atr_5': None, 'macd_hist': None, 'vwap': 100.3175, 'price_vs_vwap': -0.0024}
01-02 09:34 {'log_return': 0.0036, 'ma_3': 100.3732, 'rsi_5': None, 'atr_5': 0.3482, 'macd_hist': None, 'vwap': 100.3329, 'price_vs_vwap': 0.0011}
01-02 09:35 {'log_return': 0.0018, 'ma_3': 100.3797, 'rsi_5': 78.3739, 'atr_5': 0.3622, 'macd_hist': None, 'vwap': 100.3747, 'price_vs_vwap': 0.0024}


### Online equals batch

The online accumulators are written to reproduce the batch math exactly. The
`stream_frame` helper replays a frame bar by bar; here we confirm it matches
`compute_features` to floating-point tolerance across every feature.

In [9]:
online_features = OnlineFeatureEngine(live_config).stream_frame(cleaned)
batch_features = compute_features(cleaned, live_config)

for column in batch_features.columns:
    if column in ("symbol", "ts"):
        continue
    np.testing.assert_allclose(
        online_features[column].to_numpy(float),
        batch_features[column].to_numpy(float),
        rtol=1e-9, atol=1e-9, equal_nan=True,
    )
print("online stream == batch for all features: OK")
online_features.head(8)

online stream == batch for all features: OK


,symbol,ts,log_return,ma_3,rsi_5,atr_5,macd_hist,vwap,price_vs_vwap
0,AAPL,2024-01-02 09:30:00,NaN,NaN,NaN,NaN,NaN,100.157361,-0.000190
1,AAPL,2024-01-02 09:31:00,0.003286,NaN,NaN,NaN,NaN,100.322121,0.001454
2,AAPL,2024-01-02 09:32:00,0.001322,100.402385,NaN,NaN,NaN,100.405141,0.001949
3,AAPL,2024-01-02 09:33:00,-0.005213,100.382216,NaN,NaN,NaN,100.317458,-0.002389
4,AAPL,2024-01-02 09:34:00,0.003621,100.373194,NaN,0.348234,NaN,100.332888,0.001077
5,AAPL,2024-01-02 09:35:00,0.001785,100.379710,78.373851,0.362223,NaN,100.374715,0.002448
6,AAPL,2024-01-02 09:36:00,-0.002148,100.488609,66.738607,0.399289,-0.013961,100.379170,0.000253
7,AAPL,2024-01-02 09:37:00,0.002324,100.554367,72.301886,0.415859,0.004998,100.397155,0.002401


## 8) Where to go next

- **Add a feature:** put the function in the matching `features/*.py`, decorate
  with `@register(...)`, add a `[[features.params]]` entry in `config.toml`. For
  live use, add an O(1) accumulator in `engine/online.py` and the equivalence
  test will check it against the batch version.
- **Run the pipeline:** `uv run python run.py --config config.toml`
- **Run tests:** `uv run pytest -q`

Key idea: the batch features are the single source of truth for the math, and the
online engine is held to that truth by the equivalence tests in
`tests/test_engines.py`.